# 04 - Inference and API Test

Validar predicción local y endpoint `/predict` de FastAPI.

## Prerrequisitos

- Tener `artifacts/best_model.pth` disponible.
- Tener FastAPI corriendo con `uv run uvicorn api.main:app --reload` para probar el endpoint.

In [ ]:
from pathlib import Path
import sys

import requests
from PIL import Image

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from real_estate_ml.inference.predictor import Predictor

In [ ]:
predictor = Predictor(
    checkpoint_path=ROOT / "artifacts" / "best_model.pth",
    backbone="efficientnet_b0",
    num_classes=15,
    device="cuda",
)
print("Predictor ready")

In [ ]:
sample_candidates = list((ROOT / "data" / "processed" / "test").glob("*/*.jpg"))
if not sample_candidates:
    sample_candidates = list((ROOT / "data" / "processed" / "test").glob("*/*.png"))

sample_path = sample_candidates[0]
image = Image.open(sample_path).convert("RGB")
print(sample_path)
image

In [ ]:
predictor.predict(image, top_k=3)

In [ ]:
API_URL = "http://127.0.0.1:8000/predict"
with open(sample_path, "rb") as f:
    response = requests.post(API_URL, files={"file": (sample_path.name, f, "image/jpeg")}, timeout=30)

response.status_code, response.json()

## Checklist (merged from `inference_walkthrough.ipynb`)

Validaciones clave de inferencia:

1. Cargar `artifacts/best_model.pth` con `Predictor`.
2. Probar prediccion local (`predictor.predict`) sobre una imagen real.
3. Probar endpoint `POST /predict` con la misma imagen.
4. Comparar que respuesta API y prediccion local sean coherentes.

Troubleshooting rapido:
- Si falla carga de modelo, comprobar ruta de checkpoint.
- Si falla API, verificar que `uvicorn` este levantado y responda en `/docs`.
- Si no hay imagen de prueba, revisar `data/processed/test`.